# 01 — تشخیص Intent با مدل زبانی (LLM)

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("."))
from chatbot_common import (
    call_llm, extract_json, load_intents, load_fewshot, load_raw_examples,
    DATA_DIR, VAL_DIR, INTENT_MAPPING_CSV,
)

INTENTS = load_intents()
FEW_SHOT = load_fewshot()
print(f"{len(INTENTS)} intent بارگذاری شد.")
print(f"few-shot برای {len(FEW_SHOT)} intent موجود است." if FEW_SHOT else "few-shot موجود نیست -> zero-shot")


## ساخت System Prompt

In [ ]:
INTENT_DESCRIPTIONS = {
    "openـaccountـfree": "افتتاح حساب رایگان/قرض‌الحسنه",
    "openـaccountـcurrent": "افتتاح حساب جاری",
    "openـaccountـdeposit": "افتتاح حساب سپرده/سرمایه‌گذاری",
    "loanـfree": "درخواست وام قرض‌الحسنه (بدون سود)",
    "loanـinterest": "درخواست وام با سود/بهره",
    "card2card": "انتقال وجه کارت به کارت",
    "paya": "انتقال وجه پایا/ساتنا با شماره شبا",
    "convertـcheque": "نقد کردن یا تبدیل چک",
    "receiptـpayment": "پرداخت قبض یا فاکتور",
    "installmentـpayment": "پرداخت قسط وام",
    "turnoverـbill": "دریافت گردش/صورت‌حساب تراکنش‌ها",
    "balanceـbill": "استعلام موجودی حساب",
    "submitـcheque": "ثبت/صدور چک جدید",
    "receiveـcheque": "دریافت وجه چک",
    "changeـpassword": "تغییر رمز عبور کارت",
    "duplicateـcard": "درخواست کارت المثنی/تکراری",
    "closeـcard": "مسدود کردن کارت",
    "delegateـaccount": "وکالت دادن حساب به شخص دیگر",
    "currencyـrequest": "درخواست خرید/تبدیل ارز",
    "softwareـproblem": "گزارش مشکل نرم‌افزاری همراه‌بانک",
    "signinـproblem": "مشکل ورود به حساب کاربری",
}

def build_intent_system_prompt():
    lines = [
        "شما یک دستیار طبقه‌بندی قصد (intent classification) برای یک چت‌بات بانکی فارسی هستید.",
        "وظیفه‌ی شما فقط تشخیص intent پیام کاربر از میان لیست زیر است؛ اسلات‌ها یا اطلاعات دیگر را استخراج نکنید.",
        "",
        "لیست intent های مجاز به‌همراه توضیح کوتاه:",
    ]
    for intent in INTENTS:
        desc = INTENT_DESCRIPTIONS.get(intent, "")
        lines.append(f"- {intent}: {desc}")

    if FEW_SHOT:
        lines.append("")
        lines.append("چند نمونه:")
        for intent, examples in FEW_SHOT.items():
            for ex in examples[:1]:
                lines.append(f'- متن: "{ex["text"]}" -> intent: {intent}')

    lines.append("")
    lines.append("فقط و فقط یک شیء JSON با این فرمت دقیق برگردانید، بدون هیچ متن اضافه یا Markdown:")
    lines.append('{"intent": "<یکی از intent های بالا>", "confidence": "high|medium|low"}')
    lines.append("اگر پیام کاربر با هیچ‌کدام از intent های بالا مطابقت نداشت، مقدار intent را \"unknown\" بگذارید.")
    return "\n".join(lines)

INTENT_SYSTEM_PROMPT = build_intent_system_prompt()
print(INTENT_SYSTEM_PROMPT[:600] + "\n...")


## تابع تشخیص Intent

In [ ]:
import difflib

def classify_intent(user_text: str) -> dict:
    """برمی‌گرداند: {'intent': str, 'confidence': str, 'raw': str}"""
    raw = call_llm(INTENT_SYSTEM_PROMPT, user_text, max_tokens=200, temperature=0.0)
    try:
        parsed = extract_json(raw)
        intent = parsed.get("intent", "unknown")
        confidence = parsed.get("confidence", "unknown")
    except Exception:
        intent, confidence = "unknown", "low"

    if intent not in INTENTS and intent != "unknown":
        close = difflib.get_close_matches(intent, INTENTS, n=1, cutoff=0.6)
        intent = close[0] if close else "unknown"

    return {"intent": intent, "confidence": confidence, "raw": raw}


## تست سریع

In [ ]:
demo_sentences = [
    "می‌خوام ۵۰۰ هزار تومان به کارت دوستم انتقال بدم",
    "موجودی حسابم چقدره؟",
    "می‌خوام قبض برق رو پرداخت کنم",
    "رمز کارتم رو یادم رفته، می‌خوام عوضش کنم",
    "برای وام با سود کم چیکار باید بکنم؟",
]

for s in demo_sentences:
    result = classify_intent(s)
    print(f"{s!r:55} -> {result['intent']}  (confidence={result['confidence']})")


## ارزیابی روی داده‌ی Validation 

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

SAMPLE_SIZE = 100  

raw_val = load_raw_examples(VAL_DIR)
if raw_val:
    intent_code_to_name = {}
    if os.path.exists(INTENT_MAPPING_CSV):
        mdf = pd.read_csv(INTENT_MAPPING_CSV)
        intent_code_to_name = dict(zip(mdf["code"], mdf["intent"]))

    examples = raw_val if SAMPLE_SIZE is None else raw_val[:SAMPLE_SIZE]

    y_true, y_pred = [], []
    for ex in tqdm(examples):
        gold_intent = intent_code_to_name.get(ex.get("intent_id"), ex.get("intent_id"))
        pred = classify_intent(ex["input_text"])["intent"]
        y_true.append(gold_intent)
        y_pred.append(pred)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print("داده‌ای پیدا نشد .")
